# Ekman advection of single years

May Ekman advection of each single year 2023–2025, from the per-year pentad files (`ekman_mse.jl 2023 2025 peryear`), compared with the multi-year May climatology. Files are chosen in **Data** by `composite_file(stem, years)`: `y:y` for a single year, `2012:2025` for the climatology.

Libraries (`composites.jl`, `maps.jl`), loaded with Revise `includet`, and the kernel ("Julia") are as in `ekman_plots.ipynb`. Run the cells in order.

In [ ]:
using Revise, Dates
includet("composites.jl") # reading composites, computing fields
includet("maps.jl")       # plotting

# display settings, which depend on the platform; change them here if figures look wrong or hang
plot_defaults!(; dpi=200) # inline figure size on screen (inches x dpi); fonts=[...]: font list
PythonPlot.svg(false)     # true: inline SVG, which scales to the pane (maps are rasterized, so it stays small)

## Data
May: the 6 pentads best aligned with May (pentads 25:30, May 1–30; year ignored in the dates) of each single year in `years1`, and of the climatology over `years`, in the region `lonlim`, `latlim`.

In [ ]:
years = 2012:2025  # period of the climatology
years1 = 2023:2025 # single years
lonlim, latlim = (40, 110), (0, 25) # degrees: Indian Ocean; only this region is read
Ro_rad = 250e3 # m, curtail M by r = 1 - exp(-y^2/Ro_rad^2) near the equator (plotting only)
may = (Date(2001, 5, 1), Date(2001, 5, 31)) # 6 pentads best aligned with these dates

file_clim = composite_file("ekman_pentad_clim_oisst", years) # in ../out
file_year = Dict(y => composite_file("ekman_pentad_clim_oisst", y:y) for y in years1)
f_clim = ekman_fields(file_clim, may...; n=6, lonlim, latlim, Ro_rad)
f_year = Dict(y => ekman_fields(file_year[y], may...; n=6, lonlim, latlim, Ro_rad) for y in years1)
advsum(f) = f.adv_o .+ f.adv_a # ocean + atmosphere Ekman advection, W m^-2
f_clim.dimval, basename(file_clim), [basename(file_year[y]) for y in years1]

## Ocean and atmosphere Ekman advection, their sum, and its anomaly
Rows: years. Columns: ocean and atmosphere Ekman advection of (moist) enthalpy, their sum, and the sum minus the climatology's.

In [ ]:
Hlim, anomlim = 60, 30 # color limits: ±W m^-2 (advection), ±W m^-2 (anomaly of the sum)
fig_yr, axs_yr, meshes_yr, cbars_yr = plot_panels(f_clim, [
    p for y in years1 for p in (
        panel(f_year[y].adv_o, "$y ocean advection " * raw"$-M_o \cdot \nabla h_o$", flux_units; vmin=-Hlim, vmax=Hlim),
        panel(f_year[y].adv_a, "$y atmosphere advection " * raw"$-M_a \cdot \nabla h_a$", flux_units; vmin=-Hlim, vmax=Hlim),
        panel(advsum(f_year[y]), "$y oc+atm advection sum", flux_units; vmin=-Hlim, vmax=Hlim),
        panel(advsum(f_year[y]) .- advsum(f_clim), "$y sum minus $(first(years))-$(last(years))", flux_units;
              vmin=-anomlim, vmax=anomlim))];
    ncols=4, figsize=(22, 3.2length(years1)), ylim=latlim,
    suptitle="single years, $(period_label(f_clim.dimval)) (Ro_rad = $(Ro_rad/1e3) km)");
# fig_yr.savefig("ekman_adv_years_$(first(years1))-$(last(years1))_$(period_tag(f_clim.dimval)).png"; bbox_inches="tight")